In [ ]:
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from pathlib import Path
import pandas as pd
from src.utils import split_dataframe, evaluate_baseline
import os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report
import joblib
from scipy.sparse import vstack

tqdm.pandas()
load_dotenv()
SEED = int(os.getenv("SEED", "42"))

## TF-IDF alapú LinearSVC tanítása gyakori ICD-10-CM chapter osztályozásra (Training a TF-IDF-based LinearSVC for frequent ICD-10-CM chapter classification)

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "frequent_chapter/without_dropped_sections",
    processed_data_dir / "frequent_chapter/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")
    
    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))        
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_base_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _  = split_dataframe(df, "chapter", "subject_id", 10, SEED)

In [ ]:
X_train_text = df_train["text"]
y_train = mlb.transform(df_train["chapter"])

X_val_text = df_val["text"]
y_val = mlb.transform(df_val["chapter"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["chapter"])

In [ ]:
split_index = [-1] * len(X_train_text) + [0] * len(X_val_text)
pds = PredefinedSplit(test_fold=split_index)

X_combined = pd.concat([X_train_text, X_val_text], axis=0)
y_combined = vstack([y_train, y_val])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

In [ ]:
parameters = {
    'tfidf__max_features': [5000, 7500, 10000],
    'tfidf__ngram_range': [(1, 1),(1, 2)],
    'tfidf__max_df': [0.8, 0.9],
    'tfidf__min_df': [0.001, 0.01],
    'clf__estimator__C': [0.1, 1],
    'clf__estimator__class_weight': [None,"balanced"],
    'tfidf__sublinear_tf': [True,False]
}

In [ ]:
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=pds,  
    scoring='f1_micro',
    refit=False,
    n_jobs=6,
    verbose=3
)

In [ ]:
grid_search.fit(X_combined, y_combined)

In [ ]:
print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest score:")
print(grid_search.best_score_)

In [ ]:
final_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

final_model.set_params(**grid_search.best_params_)

In [ ]:
final_model.fit(X_train_text, y_train)

In [ ]:
eval_dir = baseline_model_path / "val_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report = evaluate_baseline(final_model, X_val_text, y_val, mlb)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
eval_dir = baseline_model_path / "test_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report = evaluate_baseline(final_model, X_test_text, y_test, mlb)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
best_model_dir = baseline_model_path / "best_model"
best_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, best_model_dir / "pipeline.joblib")

joblib.dump(mlb, best_model_dir / "mlb.joblib")

## TF-IDF alapú LinearSVC tanítása top 50 ICD-10-CM kód osztályozásra (Training a TF-IDF-based LinearSVC for top 50 ICD-10-CM code classification)

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "top_50_code/without_dropped_sections",
    processed_data_dir / "top_50_code/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")

    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_base_dataset.parquet")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _ = split_dataframe(df, "icd_code", "subject_id", 10, SEED)

In [ ]:
X_train_text = df_train["text"]
y_train = mlb.transform(df_train["icd_code"])

X_val_text = df_val["text"]
y_val = mlb.transform(df_val["icd_code"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["icd_code"])

In [ ]:
split_index = [-1] * len(X_train_text) + [0] * len(X_val_text)
pds = PredefinedSplit(test_fold=split_index)

X_combined = pd.concat([X_train_text, X_val_text], axis=0)
y_combined = vstack([y_train, y_val])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

In [ ]:
parameters = {
    'tfidf__max_features': [5000, 7500, 10000],
    'tfidf__ngram_range': [(1, 1),(1, 2)],
    'tfidf__max_df': [0.8, 0.9],
    'tfidf__min_df': [0.001, 0.01],
    'clf__estimator__C': [0.1, 1],
    'clf__estimator__class_weight': [None,"balanced"],
    'tfidf__sublinear_tf': [True,False]
}

In [ ]:
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=pds,  
    scoring='f1_micro',
    refit=False,
    n_jobs=6,
    verbose=3
)

In [ ]:
grid_search.fit(X_combined, y_combined)

In [ ]:
print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest score:")
print(grid_search.best_score_)

In [ ]:
final_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', OneVsRestClassifier(LinearSVC(max_iter=5000, random_state=SEED)))
])

final_model.set_params(**grid_search.best_params_)

In [ ]:
final_model.fit(X_train_text, y_train)

In [ ]:
eval_dir = baseline_model_path / "val_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report = evaluate_baseline(final_model, X_val_text, y_val, mlb)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
eval_dir = baseline_model_path / "test_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report = evaluate_baseline(final_model, X_test_text, y_test, mlb)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
best_model_dir = baseline_model_path / "best_model"
best_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, best_model_dir / "pipeline.joblib")

joblib.dump(mlb, best_model_dir / "mlb.joblib")